# LangChain: Evaluation

## Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation
* LangChain evaluation platform

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

This notebook uses **Anthropic Claude Haiku** (`ChatAnthropic`) and **Hugging Face** embeddings locally (`all-MiniLM-L6-v2`). Set `ANTHROPIC_API_KEY` in your `.env`. The course video uses OpenAI; evaluation concepts are the same.

**LangChain v1:** install **`langchain-classic`** for `RetrievalQA`, `VectorstoreIndexCreator`, and evaluation chains (imports use `langchain_classic.*`).

> **If you see `NotFoundError` / 404 for `model:`:** update `llm_model` to a current Haiku id from [Anthropic’s models page](https://docs.anthropic.com/en/docs/about-claude/models).

In [2]:
# Claude Haiku — same default as other notebooks in this repo
llm_model = "claude-haiku-4-5-20251001"

## Create our QandA application

In [3]:
# pip install langchain-classic langchain-anthropic langchain-community sentence-transformers
from langchain_classic.chains import RetrievalQA
from langchain_anthropic import ChatAnthropic
from langchain_community.document_loaders import CSVLoader
from langchain_classic.indexes import VectorstoreIndexCreator
from langchain_community.vectorstores import DocArrayInMemorySearch
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/Users/trentoncreamer/Crypto/DataCore Labs/Training/LangChain/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/rr/95dx0y9j0ljcdfpq4mb2ds5w0000gn/T/ipykernel_89899/562168261.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8087.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
-----------

In [8]:
from pathlib import Path

# CSV is under vectors_embeddings/; cwd may be project root or evaluation/
for candidate in (
    Path("OutdoorClothingCatalog_1000.csv"),
    Path("evaluation") / "OutdoorClothingCatalog_1000.csv",
    Path("vectors_embeddings") / "OutdoorClothingCatalog_1000.csv",
    Path("..") / "vectors_embeddings" / "OutdoorClothingCatalog_1000.csv",
):
    if candidate.is_file():
        file = str(candidate.resolve())
        break
else:
    raise FileNotFoundError(
        "OutdoorClothingCatalog_1000.csv not found. "
        "Copy it from vectors_embeddings/ next to this notebook, or run with project root as cwd."
    )

loader = CSVLoader(file_path=file)
data = loader.load()

In [9]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embeddings,
).from_loaders([loader])

KeyboardInterrupt: 

In [ ]:
llm = ChatAnthropic(temperature=0.0, model=llm_model, max_tokens=4096)
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=index.vectorstore.as_retriever(), 
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>"
    }
)

### Coming up with test datapoints

In [ ]:
data[10]

In [ ]:
data[11]

### Hard-coded examples

In [ ]:
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

### LLM-Generated examples

In [ ]:
from langchain_classic.evaluation.qa import QAGenerateChain


In [ ]:
example_gen_chain = QAGenerateChain.from_llm(
    ChatAnthropic(temperature=0, model=llm_model, max_tokens=4096)
)

In [ ]:
# the warning below can be safely ignored

In [ ]:
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

In [ ]:
new_examples[0]

In [ ]:
data[0]

### Combine examples

In [ ]:
examples += new_examples

In [ ]:
qa.run(examples[0]["query"])

## Manual Evaluation

In [ ]:
import langchain
langchain.debug = True

In [ ]:
qa.run(examples[0]["query"])

In [ ]:
# Turn off the debug mode
langchain.debug = False

## LLM assisted evaluation

In [ ]:
predictions = qa.apply(examples)

In [ ]:
from langchain_classic.evaluation.qa import QAEvalChain

In [ ]:
llm = ChatAnthropic(temperature=0, model=llm_model, max_tokens=4096)
eval_chain = QAEvalChain.from_llm(llm)

In [ ]:
graded_outputs = eval_chain.evaluate(examples, predictions)

In [ ]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + predictions[i]['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Predicted Grade: " + graded_outputs[i]['text'])
    print()

In [ ]:
graded_outputs[0]

Reminder: Download your notebook to you local computer to save your work.